# 01 — EDA i kontrakt danych (Sprint 1)

Cel notebooka: poznać dane wybranego datasetu i podjąć decyzje, które trafią do kodu:

- **input normalization** — reguły w `normalize_input()` (`src/asi_project/data.py`),
- **training-only filtering** — reguły w `filter_training_rows()` (jeśli są potrzebne),
- **typy cech** — listy `features.numeric` i `features.categorical` w `project.yaml`,
- **metryka główna** — uzasadnienie w karcie projektu w README.

Jak pracować:

- uruchamiajcie notebook od góry (Restart & Run All); po zmianie `project.yaml` uruchomcie
  ponownie komórkę ustawień,
- komórki oznaczone **TODO** uzupełnia zespół; odpowiedzi tekstowe wpisujcie w miejsce
  `*(odpowiedź)*`,
- nie zmieniajcie kontraktu prowadzącej (`src/asi_project/datasets.yaml`).

## 0. Ustawienia

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

from asi_project.config import get_contract, load_project_config
from asi_project.data import (
    encode_target,
    feature_list_problems,
    filter_training_rows,
    is_text_column,
    load_dataset,
    normalize_input,
    target_classes,
    validate,
)

config = load_project_config()
contract = get_contract(config)
DATA_MODE = os.environ.get("ASI_DATA_MODE", "full")

print(f"Dataset: {contract.title} ({contract.dataset_id})")
print(f"Target: {contract.target} | klasa pozytywna: {contract.positive_class}")
print(f"Moment predykcji: {contract.prediction_moment}")

## 1. Wczytanie danych

Pełne dane pobiera `python scripts/download_data.py`. Nazwy kolumn są ujednolicane przy
wczytywaniu (białe znaki), wartości — jeszcze nie.

In [ ]:
df_raw = load_dataset(config, contract, mode=DATA_MODE)
print(df_raw.shape)
df_raw.head()

## 2. Inspekcja: typy, braki, liczba unikalnych wartości

In [ ]:
overview = pd.DataFrame(
    {
        "dtype": df_raw.dtypes.astype(str),
        "braki": df_raw.isna().sum(),
        "unikalne": df_raw.nunique(),
    },
)
overview

**TODO — obserwacje (2–4 zdania):** które kolumny mają nieoczekiwany typ? Gdzie są braki?
Które kolumny liczbowe są w rzeczywistości kategoriami?

Brak braków danych we wszystkich kolumnach. Month i VisitorType są tekstowe, Weekend i Revenue logiczne, a Month ma tylko 10 wartości (brak dwóch miesięcy). OperatingSystems, Browser, Region i TrafficType są zapisane jako int64, ale to kody kategorii, więc traktujemy je jako cechy kategoryczne.

## 3. Kontrakt danych

Kontrakt prowadzącej określa target, klasę pozytywną, ALLOWED FEATURES (jedyne kolumny
dozwolone w X) i EXCLUDED FEATURES wraz z powodem wykluczenia.

In [ ]:
problems = validate(df_raw, contract)
print("Kontrakt danych:", "OK" if not problems else problems)
print("ALLOWED FEATURES:", list(contract.allowed_features))
print("EXCLUDED FEATURES:")
for column, reason in contract.excluded_features.items():
    status = "jest w danych" if column in df_raw.columns else "brak w pliku"
    print(f"- {column} ({status}): {reason}")

## 4. Target i klasa pozytywna

In [ ]:
print("Klasy targetu:", target_classes(df_raw[contract.target]))
y = encode_target(df_raw[contract.target].dropna(), contract.positive_class)
print(f"Udział klasy pozytywnej: {y.mean():.3f}")
ax = y.value_counts().sort_index().plot(kind="bar", title="Target (1 = klasa pozytywna)")
ax.set_xlabel("klasa")
plt.show()

**TODO — metryka główna (2–3 zdania):** którą metrykę (accuracy, precision, recall, F1)
wybieracie jako główną i dlaczego, biorąc pod uwagę udział klasy pozytywnej i koszt błędów
w waszym problemie? To samo uzasadnienie wpiszcie do karty projektu w README.

Wybieramy F1 dla klasy pozytywnej, bo zakup kończy tylko 15,5% sesji i model zawsze przewidujący brak zakupu miałby accuracy ok. 84,5%, nie znajdując żadnego kupującego. Niepotrzebny rabat obniża marżę, a pominięty klient może odejść bez zakupu, dlatego F1 łączy precision i recall.

## 5. EDA

**TODO:** co najmniej dwie analizy — jedna dla cechy numerycznej, jedna dla kategorycznej.
Wybierajcie wyłącznie spośród ALLOWED FEATURES.

In [ ]:
# TODO (S1): rozkład wybranej cechy numerycznej w podziale na klasy targetu, np.
df_raw.groupby(contract.target)["ExitRates"].describe()

In [ ]:
# TODO (S1): udział klasy pozytywnej w kategoriach wybranej cechy kategorycznej, np.
pd.crosstab(df_raw["Month"], df_raw[contract.target], normalize="index").sort_values(
    True, ascending=False
)

**TODO — wnioski z EDA (3–5 zdań):**

Sesje zakończone zakupem mają ponad dwukrotnie niższy ExitRates (średnia 0,020 wobec 0,047, mediana 0,016 wobec 0,029), więc ta cecha dobrze różnicuje klasy. Rozkład ExitRates jest prawoskośny, a w obu klasach występuje maksimum 0,2, dlatego cechy numeryczne skalujemy w Pipeline. Udział zakupów silnie zależy od miesiąca: od 25,4% w Nov do 1,6% w Feb, więc Month zostaje cechą kategoryczną. Month ma niespójny zapis (June zamiast Jun), co ujednolicimy w normalize_input, a brak Jan i Apr wymaga OneHotEncoder z handle_unknown="ignore".

## 6. Input normalization

Input normalization działa na pojedynczym rekordzie, nie używa targetu ani statystyk zbioru
i jest idempotentna. Typowe kandydatki: białe znaki, puste napisy, oczywiste reprezentacje
braków, ujednolicenie zapisu wartości. Uzupełnianie braków medianą **nie jest** input
normalization — wymaga `fit`, więc należy do model preprocessing.

In [ ]:
text_columns = [column for column in df_raw.columns if is_text_column(df_raw[column])]
for column in text_columns:
    values = df_raw[column].dropna().astype(str)
    padded = int((values != values.str.strip()).sum())
    empty = int((values.str.strip() == "").sum())
    top = values.value_counts().head(6).to_dict()
    print(f"{column}: spacje na brzegach={padded}, puste={empty}, najczęstsze={top}")

**TODO — reguły input normalization:** jakie reguły dopisujecie do `normalize_input()` i dlaczego
są bezpieczne przy predykcji pojedynczego rekordu? Po zmianie funkcji uruchomcie `pytest`.

W normalize_input() dopisujemy trzy reguły: usunięcie białych znaków na brzegach w kolumnach tekstowych, zamianę pustych napisów na brak wartości (NaN) oraz ujednolicenie zapisu miesiąca June na Jun, zgodnie z pozostałymi skrótami. Reguły są bezpieczne przy predykcji pojedynczego rekordu, bo każda działa tylko na wartości w danym wierszu, nie korzysta z targetu ani statystyk zbioru i jest idempotentna. Obecnie dane nie mają spacji ani pustych wartości, ale te reguły chronią przed nimi na wejściu API.

In [ ]:
normalized = normalize_input(df_raw)
changed = (normalized.astype(str) != df_raw.astype(str)).sum()
changed[changed > 0]

## 7. Training-only filtering

Training-only filtering działa tylko przy przygotowaniu danych treningowych i może jedynie
usuwać rekordy. Domyślnie `filter_training_rows()` usuwa rekordy bez targetu.

In [ ]:
print("Rekordy bez targetu:", int(df_raw[contract.target].isna().sum()))
print("Dokładne duplikaty wierszy:", int(df_raw.duplicated().sum()))
filtered = filter_training_rows(normalize_input(df_raw), contract)
print(f"Po training-only filtering: {len(filtered)} z {len(df_raw)} rekordów")

**TODO — decyzja o training-only filtering:** czy usuwacie dodatkowo jakieś rekordy
(np. dokładne duplikaty)? Dlaczego tak albo dlaczego nie?

Tak, dodatkowo usuwamy 125 dokładnych duplikatów (ok. 1% danych). To głównie krótkie sesje bez zakupu, które sztucznie zwiększają wagę tych samych wzorców, a po podziale ten sam wiersz mógłby trafić jednocześnie do zbioru treningowego i testowego. Usuwamy je tylko przy przygotowaniu danych treningowych, a przy predykcji każdy rekord dostaje odpowiedź.

## 8. Typy cech → `project.yaml`

**TODO:** wpiszcie do `project.yaml` listy `features.numeric` i `features.categorical`
(wyłącznie ALLOWED FEATURES), zapiszcie plik i uruchomcie ponownie komórkę ustawień oraz
komórkę poniżej.

In [ ]:
print("Numeryczne:", list(config.numeric_features))
print("Kategoryczne:", list(config.categorical_features))
print("Problemy z deklaracją:", feature_list_problems(contract, config.model_features) or "brak")
print("Kontrakt z deklaracją:", validate(normalize_input(df_raw), contract, config) or "OK")

## 9. Podsumowanie decyzji

**TODO — w 3–5 punktach:** decyzje o input normalization, training-only filtering, typach cech
i metryce głównej. Te decyzje będą punktem wyjścia do ADR-1 w Sprincie 2.

1. Input normalization: czyścimy tekst (spacje, puste wartości) i poprawiamy zapis June na Jun, żeby wszystkie miesiące miały ten sam format.

2. Training-only filtering: z danych treningowych usuwamy rekordy bez targetu i 125 powtórzonych wierszy.

3. Typy cech: numery systemów, przeglądarek, regionów i źródeł ruchu to kategorie, a nie liczby, więc traktujemy je jak Month, VisitorType i Weekend. PageValues nie używamy, bo jest wykluczona przez kontrakt.

4. Przygotowanie do modelu: cechy liczbowe skalujemy, a kategorie kodujemy tak, żeby model radził sobie z nowymi wartościami (np. miesiącami, których nie ma w danych).

5. Metryka: F1 dla klasy „zakup”, bo kupujących jest tylko 15,5%. Accuracy byłaby myląca, a podział na train/test zachowuje proporcje klas.